# 07 — Stop Words
**Goal:** Understand when to remove (and NOT remove) common words.

Stop words are high-frequency function words — `the`, `and`, `of`, `with` — that carry little standalone meaning. Removing them shrinks the vocabulary, speeds up matching and TF-IDF-style pipelines, and is a standard step in classic NLP. The catch: "little standalone meaning" is context-dependent, and on resumes the context is everything.

**Why it matters for resumes / ATS:** a bullet like "Reduced costs **by** 20%" loses its metric if `by` is deleted, and "Experience **with** Python" loses the tool signal if `with` goes. Default stop lists are tuned for news articles; using them unmodified on resumes actively destroys structured information.

| Word | Role in Resume | Stop List? | Impact if Removed |
|---|---|---|---|
| `of` | Quantifies team size | Yes | "team 5 engineers" — metric lost |
| `by` | Introduces improvement | Yes | "costs 20%" — context lost |
| `with` | Introduces tool/skill | Yes | "Experience Python" — ambiguous |
| `the` | Article | Yes | Safe to remove |
| `and` | Conjunction | Yes | Safe to remove |

## 1. NLTK vs spaCy Stop Lists

Every library ships a stop list, and they disagree. NLTK's is curated from general corpora; spaCy's `Defaults.stop_words` is larger because it also covers contractions and common verb forms. Neither list was built with resumes in mind.

| Library | Count | Coverage |
|---|---|---|
| NLTK | 198 words | Curated from general corpora |
| spaCy | 326 words | Larger, includes contractions |

A 128-word gap is a real difference in downstream filtering — spaCy marks roughly 40% more tokens as removable.

In [ ]:
from nltk.corpus import stopwords
import spacy

nltk_stops = set(stopwords.words("english"))
spacy_stops = spacy.load("en_core_web_sm").Defaults.stop_words

print(f"NLTK: {len(nltk_stops)} words")
print(f"spaCy: {len(spacy_stops)} words")
print(f"Difference: {len(spacy_stops) - len(nltk_stops)} more words in spaCy")

# Show some words in spaCy but not NLTK
diff = spacy_stops - nltk_stops
print(f"\nSample words in spaCy only: {sorted(list(diff))[:10]}")

## 2. The Problem — Stop Words That Matter

Function words become load-bearing on resumes. `of` quantifies team size, `by` introduces the improvement metric, `with` introduces the tool. Deleting them by rule destroys exactly the structured facts an ATS is trying to extract.

**What the code does:** parses three bullets and collects the tokens spaCy flags as `is_stop`:

| Bullet | Stop Words Found | Why They Matter |
|---|---|---|
| "Managed team of 5 engineers" | `[of]` | Team-size signal |
| "Reduced costs by 20%" | `[by]` | Improvement metric |
| "Experience with Python" | `[with]` | Tool-usage signal |

In [ ]:
lines = [
    "Managed team of 5 engineers",   # "of" = team size signal
    "Reduced costs by 20%",         # "by" = improvement metric
    "Experience with Python",       # "with" = tool usage
]

print(f"{'Bullet':<40} {'Stop Words Found'}")
print("-" * 60)
for l in lines:
    doc = spacy.load("en_core_web_sm")(l)
    stops = [t.text for t in doc if t.is_stop]
    print(f"{l:<40} {stops}")

**Observation:** In all three cases, the "stop word" is the word that makes the bullet a measurable claim. A rule that strips stop words before matching turns these into `Managed team 5 engineers`, `Reduced costs 20%`, `Experience Python` — matchable, but semantically gutted for any structured extraction.

## 3. Custom Resume Stop Words

The fix is a two-tier custom list:

| Category | Words | Rationale |
|---|---|---|
| Generic noise | `a`, `an`, `the`, `very`, `really`, `just` | No matching value in any resume |
| Resume filler | `experienced`, `years`, `including`, `highly`, `motivated` | Self-marketing padding, no ATS keywords |

**What the code does:** defines both sets, tokenizes a phrase, and filters out every token in either set.

In [ ]:
noise = {"a", "an", "the", "and", "or", "but", "its", "very", "really", "just"}
resume_stops = {"experienced", "years", "working", "including", "various", "multiple",
                "please", "resume", "professional", "summary", "highly", "motivated"}

tokens = "Highly motivated team player with 5 years experience".lower().split()
filtered = [t for t in tokens if t not in noise and t not in resume_stops]

print(f"Before ({len(tokens)} tokens): {tokens}")
print(f"After  ({len(filtered)} tokens): {filtered}")
print(f"\nRemoved: {[t for t in tokens if t not in filtered]}")

**Try it:** note `with` survives — deliberately, because §2 showed `with` signals tool usage. The custom list encodes domain judgment: what to drop is a *product decision* about what your matcher should see, not a linguistics default.

| Token | In Noise? | In Resume Stops? | Kept? |
|---|---|---|---|
| `highly` | No | Yes | ❌ Removed |
| `motivated` | No | Yes | ❌ Removed |
| `team` | No | No | ✓ Kept |
| `player` | No | No | ✓ Kept |
| `with` | No | No | ✓ Kept (tool signal) |
| `5` | No | No | ✓ Kept |
| `years` | No | Yes | ❌ Removed |
| `experience` | No | No | ✓ Kept |

## Key Insight

**Stop-word removal is a recall trade-off, and the defaults are tuned for prose, not resumes.**

`of`, `by`, and `with` are stop words in a news corpus — and the carriers of team size, metrics, and tool usage in a resume. Removing them before matching is a one-way door: the information is gone and no later stage can recover it.

The discipline that works:
1. Keep function words that precede numbers or nouns of interest
2. Drop only words with provably no matching value
3. Test the list against real bullets before deploying

Once the noise is gone, the remaining content words still carry inflection — `develop`, `developing`, `developed` — which is exactly what Ch. 08 lemmatization normalizes next.